In [1]:
# Import required packages
import earthaccess # search and access NASA Earthdata 
import xarray as xr # load and analyze N-dimensional array data

In [2]:
# Authentication
# Note: If a strategy is not specified, environment variables will be used first, then a .netrc, and finally a user's input.
earthaccess.login() 

In [3]:
# Data search
data_doi = '10.5067/ATLAS/ATL03.007'

results = earthaccess.search_data(
    doi = data_doi, # search by dataset DOI
    temporal = ("2025-01-01", "2025-12-31"), # one week of data  
    bounding_box=(-108.3, 38.9, -107.8, 39.1),  # small area over Grand Mesa, CO, USA
)

/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/search.py:946: FutureWarning: As of version 1.0, `DataCollection.concept_id` will be accessed as an attribute; e.g. use `DataCollection.concept_id` **not** `DataCollection.concept_id()`
  concept_id = collection[0].concept_id()
/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [4]:
len(results)

7

In [5]:
files = earthaccess.open(results[:1])  # stream a single granule without downloading it
ds = xr.open_datatree(files[0], engine="h5netcdf", phony_dims="access")  # open as a tree of groups
# ds = xr.open_datatree(files[0], engine="h5netcdf")  # open as a tree of groups

/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
print(ds)

<xarray.DataTree>
Group: /
│   Dimensions:       (ds_surf_type: 5, ds_xyz: 3)
│   Coordinates:
│     * ds_surf_type  (ds_surf_type) int32 20B 1 2 3 4 5
│     * ds_xyz        (ds_xyz) int32 12B 1 2 3
│   Attributes: (12/47)
│       Conventions:                        CF-1.8
│       citation:                           Cite these data in publications as fo...
│       contributor_name:                   Thomas A Neumann (thomas.neumann@nasa...
│       contributor_role:                   Instrument Engineer, Investigator, Pr...
│       creator_name:                       GSFC I-SIPS > ICESat-2 Science Invest...
│       date_created:                       2025-05-09T17:21:05.000000Z
│       ...                                 ...
│       summary:                            The purpose of ATL03 is to provide al...
│       time_coverage_duration:             511.0
│       time_coverage_end:                  2025-01-04T10:10:24.000000Z
│       time_coverage_start:                2025-01-04T10:0

In [8]:
# Subset variables
photons = ds["gt1l/heights"].ds[["h_ph", "lat_ph", "lon_ph", "signal_conf_ph"]] # subset photon height, confidence flag, lat, lon from one beam

# Mean photon analysis for high-confidence photons
conf = photons["signal_conf_ph"].isel({photons["signal_conf_ph"].dims[1]: 0})  # pick the land column (0) from the per-surface-type confidence scores, giving one score per photon
mean_height = photons["h_ph"].where(conf >= 3).mean().item()  # mean height of medium/high-confidence photons only

print(f"Mean signal photon height: {mean_height:.1f} m")

Mean signal photon height: 1048.1 m


In [9]:
# Import required packages (pip install earthaccess xarray h5netcdf)
import earthaccess  # search and access NASA Earthdata
import xarray as xr  # load and analyze N-dimensional array data

# Authentication
earthaccess.login()  # log in with Earthdata credentials (prompts, or uses env vars/.netrc)

# Data search
data_doi = "10.5067/ATLAS/ATL03.007"  # ATL03 v7, specified by Digital Object Identifier (DOI)

results = earthaccess.search_data(
    doi=data_doi,  # search by dataset DOI
    temporal=("2025-01-01", "2025-12-31"),  # one-year temporal extent
    bounding_box=(-108.3, 38.9, -107.8, 39.1),  # small area over Grand Mesa, CO, USA
)
print(f"Granules found: {len(results)}")

# Data access
files = earthaccess.open(results[:1])  # stream the first granule without downloading it
dt = xr.open_datatree(files[0], engine="h5netcdf", phony_dims="access")  # open as a tree of groups

# Optional: download the first granule instead (ATL03 files are often several GB)
# downloaded_files = earthaccess.download(results[:1], local_path=".")
# dt = xr.open_datatree(downloaded_files[0], engine="h5netcdf", phony_dims="access")

# Inspect file contents
print(dt)

# Subset variables
photons = dt["gt1l/heights"].ds[["h_ph", "lat_ph", "lon_ph", "signal_conf_ph"]]  # photon height, lat, lon, and confidence flag from one beam

# Mean height of signal photons
conf = photons["signal_conf_ph"].isel({photons["signal_conf_ph"].dims[1]: 0})  # pick the land column (0) from the per-surface-type confidence scores, giving one score per photon
mean_height = photons["h_ph"].where(conf >= 3).mean().item()  # mean height of medium/high-confidence photons (conf >= 3)


/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/search.py:946: FutureWarning: As of version 1.0, `DataCollection.concept_id` will be accessed as an attribute; e.g. use `DataCollection.concept_id` **not** `DataCollection.concept_id()`
  concept_id = collection[0].concept_id()
/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


Granules found: 17


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

<xarray.DataTree>
Group: /
│   Dimensions:       (ds_surf_type: 5, ds_xyz: 3)
│   Coordinates:
│     * ds_surf_type  (ds_surf_type) int32 20B 1 2 3 4 5
│     * ds_xyz        (ds_xyz) int32 12B 1 2 3
│   Attributes: (12/47)
│       Conventions:                        CF-1.8
│       citation:                           Cite these data in publications as fo...
│       contributor_name:                   Thomas A Neumann (thomas.neumann@nasa...
│       contributor_role:                   Instrument Engineer, Investigator, Pr...
│       creator_name:                       GSFC I-SIPS > ICESat-2 Science Invest...
│       date_created:                       2025-05-09T17:21:05.000000Z
│       ...                                 ...
│       summary:                            The purpose of ATL03 is to provide al...
│       time_coverage_duration:             511.0
│       time_coverage_end:                  2025-01-04T10:10:24.000000Z
│       time_coverage_start:                2025-01-04T10:0